# Dark Pattern Detector — Notebook 2: Classical Model

This notebook contains the full current classical pipeline: source-aware split checks,
character 2–6 gram TF-IDF, 12 engineered features, SMOTE, LinearSVC tuning, grouped
sigmoid calibration, evaluation, and export. It does not import project scripts.

## 1. Setup

In [1]:
from pathlib import Path
import hashlib, json, re

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, PowerTransformer, RobustScaler
from sklearn.svm import LinearSVC

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
print("Project root:", ROOT)

Project root: /Users/abhigoyal/Documents/Acadss/Data Science/Projects/dark-pattern-final


## 2. Self-contained grouping and model definition

In [2]:
MODEL_NUM_COLS = ['urgency_kw_count', 'scarcity_kw_count', 'shame_phrase_flag', 'cancel_diff_score', 'social_proof_flag', 'price_drip_flag', 'discount_claim_flag', 'neg_option_flag', 'exclamation_count', 'question_count', 'number_present', 'time_reference_flag']
SLOT_VALUES = ['Tata CLiQ', 'Flipkart', 'Meesho', 'IndiaMART', 'Snapdeal', 'AJIO', 'Amazon Prime Video', 'Fitbit', 'Google Drive', 'Hostinger', 'Reliance Smart Bazaar', 'Amazon', 'Myntra', 'Croma', 'MakeMyTrip', 'BigBasket', 'JioGames', 'Paytm Money', 'sneakers', 'headphones', 'smartwatch', 'backpack', 'jacket', 'blender', 'office chair', 'yoga mat', 'coffee maker', 'desk lamp', 'phone case', 'sunglasses', 'water bottle', 'bluetooth speaker', 'air fryer', 'wallet', 'service charge', 'processing fee', 'convenience fee', 'admin surcharge', 'handling fee', 'cleanup cost', 'booking charge', 'platform fee', 'payment gateway fee', 'fulfilment fee', 'packaging fee', 'Protect Promise Fee', 'Offer Handling Fee', 'Payment Handling Fee', 'small cart charge', 'handling charge', 'delivery charge', 'pick up charge', 'surge fee', 'rain fee', 'high demand fee', 'Quick Heal', 'K7 Total Security', 'TallyPrime', 'Busy Accounting', 'Zoho Books', 'Microsoft 365', 'GST Invoice', 'Bank Statement', 'Aadhaar Copy', 'PAN Card Copy', 'Insurance Policy', 'Salary Slip', 'shipping protection', 'priority handling', 'extended warranty', 'loss protection', 'theft insurance', 'carbon offset', 'gift wrapping', 'express processing', 'damage cover', 'installation support', 'E-commerce', 'Fintech', 'Travel', 'Food Delivery', 'Streaming', 'Telecom', 'Healthcare', 'Education', '7 days', '15 days', '1 month', '3 months', '6 months', '1 year']

_VOCAB_RE = re.compile("|".join(map(re.escape, sorted(SLOT_VALUES, key=len, reverse=True))), re.I)
_CUR_RE = re.compile(r"[₹$]\s?\d[\d,]*(?:\.\d+)?")
_PCT_RE = re.compile(r"\d+\s?%")
_NUM_RE = re.compile(r"\d[\d,]*(?:\.\d+)?")

def skeleton(text):
    text = _CUR_RE.sub("<CUR>", str(text))
    text = _PCT_RE.sub("<PCT>", text)
    text = _VOCAB_RE.sub("<SLOT>", text)
    text = _NUM_RE.sub("<NUM>", text)
    return re.sub(r"\s+", " ", text).strip().lower()

def connected_groups(frame):
    parent = list(range(len(frame)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        a, b = find(a), find(b)
        if a != b:
            parent[b] = a

    for values in (frame["page_id"], frame["text"].fillna("").map(skeleton)):
        seen = {}
        for i, value in enumerate(values):
            if pd.isna(value) or not str(value).strip():
                continue
            key = str(value)
            union(i, seen[key]) if key in seen else seen.setdefault(key, i)

    return pd.factorize(np.asarray([find(i) for i in range(len(frame))]))[0]

def dataset_hash(frame):
    cols = ["page_id", "text", "label", "Pattern Category"]
    return hashlib.sha256(frame[cols].to_csv(index=False, lineterminator="\n").encode()).hexdigest()

def load_split(frame, path):
    split = json.loads(Path(path).read_text())
    train, test = np.array(split["train"]), np.array(split["test"])
    assert not set(train) & set(test)
    assert set(train) | set(test) == set(range(len(frame)))
    assert split["dataset_sha256"] == dataset_hash(frame), "stale split; rebuild it before training"
    groups = connected_groups(frame)
    assert not set(groups[train]) & set(groups[test]), "page/template leakage remains"
    return split, train, test, groups

def make_model(c=0.5):
    prep = ColumnTransformer([
        ("text", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 6), min_df=2,
                                 max_features=30_000, sublinear_tf=True), "text"),
        ("numeric", Pipeline([
            ("scale", RobustScaler()),
            ("power", PowerTransformer(method="yeo-johnson")),
        ]), MODEL_NUM_COLS),
    ])
    return ImbPipeline([
        ("prep", prep),
        ("smote", SMOTE(random_state=42)),
        ("clf", LinearSVC(C=c, random_state=42, max_iter=2_000, tol=1e-3)),
    ])

def grouped_folds(x, y, groups, n_splits=5):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    folds = list(cv.split(x, y, groups))
    classes = set(y)
    assert all(set(y[a]) == classes == set(y[b]) for a, b in folds), "a fold is missing a class"
    return folds

def calibrated_model(model, folds):
    return CalibratedClassifierCV(model, method="sigmoid", cv=folds, ensemble=False)

assert len(MODEL_NUM_COLS) == 12

## 3. Load data and verify the saved split

In [3]:
df = pd.read_csv(ROOT / "data/processed/features.csv").fillna({"text": ""})
split, train_idx, test_idx, groups = load_split(df, ROOT / "reports/leak_free_split.json")

encoder = LabelEncoder()
y = encoder.fit_transform(df["Pattern Category"])
X = df[["text"] + MODEL_NUM_COLS]
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"rows: {len(df)} | train: {len(train_idx)} | test: {len(test_idx)}")
print("grouping:", split["grouping"])
display(pd.DataFrame({"model_feature": MODEL_NUM_COLS}))

rows: 6373 | train: 5051 | test: 1322
grouping: connected_page_or_skeleton_v1


,model_feature
0,urgency_kw_count
1,scarcity_kw_count
2,shame_phrase_flag
3,cancel_diff_score
4,social_proof_flag
5,price_drip_flag
6,discount_claim_flag
7,neg_option_flag
8,exclamation_count
9,question_count


## 4. Exact model template

In [4]:
make_model(c=0.5)

,steps,"[('prep', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text', ...), ('numeric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 5. Tune, evaluate, and export

Set `RUN_TRAINING=True` to reproduce the grouped search and replace the saved model.
It is off during ordinary notebook execution so viewing the notebook stays quick.

In [5]:
RUN_TRAINING = False

if RUN_TRAINING:
    folds = grouped_folds(X_train, y_train, groups[train_idx])
    rows = []
    for c in (0.5, 1.0, 2.0):
        scores = cross_val_score(make_model(c), X_train, y_train, cv=folds,
                                 scoring="f1_macro", n_jobs=-1)
        rows.append({"C": c, "CV Macro-F1": scores.mean(), "CV Std": scores.std()})
        print(f"C={c}: {scores.mean():.4f} +/- {scores.std():.4f}")

    comparison = pd.DataFrame(rows).sort_values(
        ["CV Macro-F1", "CV Std"], ascending=[False, True]
    ).reset_index(drop=True)
    best_c = float(comparison.iloc[0]["C"])
    model = calibrated_model(
        make_model(best_c),
        grouped_folds(X_train, y_train, groups[train_idx], n_splits=3),
    ).fit(X_train, y_train)
    pred = model.predict(X_test)
    test_f1 = f1_score(y_test, pred, average="macro", zero_division=0)
    test_acc = accuracy_score(y_test, pred)
    print(f"test macro-F1={test_f1:.4f} | accuracy={test_acc:.4f}")

    report = classification_report(y_test, pred, target_names=encoder.classes_, zero_division=0)
    (ROOT / "reports/test_classification_report.txt").write_text(
        f"Held-out test accuracy: {test_acc:.4f}\nHeld-out test macro-F1: {test_f1:.4f}\n\n{report}"
    )
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=encoder.classes_,
                                             xticks_rotation=90, cmap="viridis")
    plt.gcf().set_size_inches(11, 9); plt.tight_layout()
    plt.savefig(ROOT / "reports/confusion_matrix.png", dpi=120); plt.close()

    ood_f1 = None
    ood_path = ROOT / "data/processed/ood_features.csv"
    if ood_path.exists():
        ood = pd.read_csv(ood_path)
        known = ood["Pattern Category"].isin(encoder.classes_)
        ood_y = encoder.transform(ood.loc[known, "Pattern Category"])
        ood_pred = model.predict(ood.loc[known, ["text"] + MODEL_NUM_COLS])
        ood_f1 = f1_score(ood_y, ood_pred, average="macro", zero_division=0)

    deployment = calibrated_model(
        make_model(best_c), grouped_folds(X, y, groups, n_splits=3)
    ).fit(X, y)
    joblib.dump(deployment, ROOT / "models/best_multi_model.joblib")
    joblib.dump(encoder, ROOT / "models/label_encoder.joblib")
    comparison.to_csv(ROOT / "reports/cv_model_comparison.csv", index=False)

    metrics = {
        "model": "character TF-IDF + 12 engineered features + SMOTE + calibrated LinearSVC",
        "n_rows": len(df), "n_classes": len(encoder.classes_),
        "split_grouping": split["grouping"], "test_accuracy": test_acc,
        "test_macro_f1": test_f1, "ood_dev_macro_f1": ood_f1,
        "engineered_features": MODEL_NUM_COLS,
        "best_params": {"C": best_c, "ngram_range": [2, 6]},
        "best_cv_macro_f1": float(comparison.iloc[0]["CV Macro-F1"]),
        "best_cv_std": float(comparison.iloc[0]["CV Std"]),
        "cv_comparison": comparison.to_dict(orient="records"),
    }
    (ROOT / "reports/metrics_summary.json").write_text(json.dumps(metrics, indent=2))
else:
    print("Using saved results. Set RUN_TRAINING=True to rerun training and export.")

Using saved results. Set RUN_TRAINING=True to rerun training and export.


## 6. Stored grouped results

In [6]:
metrics = json.loads((ROOT / "reports/metrics_summary.json").read_text())

# Cheap refresh: score the already-exported deployment artifact; do not retrain.
deployment = joblib.load(ROOT / "models/best_multi_model.joblib")
saved_encoder = joblib.load(ROOT / "models/label_encoder.joblib")
ood = pd.read_csv(ROOT / "data/processed/ood_features.csv")
known = ood["Pattern Category"].isin(saved_encoder.classes_)
ood_true = ood.loc[known, "Pattern Category"].to_numpy()
ood_pred = saved_encoder.inverse_transform(
    deployment.predict(ood.loc[known, ["text"] + MODEL_NUM_COLS])
)

artifact_ood_f1 = f1_score(ood_true, ood_pred, average="macro", zero_division=0)
artifact_ood_acc = accuracy_score(ood_true, ood_pred)

display(pd.DataFrame(metrics["cv_comparison"]).round(4))
display(pd.Series({
    "model": metrics["model"],
    "test accuracy": metrics["test_accuracy"],
    "test macro-F1": metrics["test_macro_f1"],
    "exported artifact OOD rows": int(known.sum()),
    "exported artifact OOD classes": int(ood.loc[known, "Pattern Category"].nunique()),
    "exported artifact OOD accuracy": artifact_ood_acc,
    "exported artifact OOD macro-F1": artifact_ood_f1,
}))

,C,CV Macro-F1,CV Std
0,0.5,0.7387,0.0593
1,1.0,0.7377,0.0670
2,2.0,0.7339,0.0672


,0
model,character TF-IDF + 12 engineered features + SMOTE + calibrated LinearSVC
test accuracy,0.824508
test macro-F1,0.731596
exported artifact OOD rows,28
exported artifact OOD classes,9
exported artifact OOD accuracy,0.892857
exported artifact OOD macro-F1,0.752381


## 7. Load the exported artifact

In [7]:
model = joblib.load(ROOT / "models/best_multi_model.joblib")
label_encoder = joblib.load(ROOT / "models/label_encoder.joblib")
sample = df.iloc[[0]][["text"] + MODEL_NUM_COLS]
probabilities = model.predict_proba(sample)[0]
winner = int(probabilities.argmax())
print("pipeline steps:", list(model.calibrated_classifiers_[0].estimator.named_steps))
print("prediction:", label_encoder.inverse_transform([winner])[0])
print("calibrated probability:", round(float(probabilities[winner]), 4))

pipeline steps: ['prep', 'smote', 'clf']
prediction: False Urgency
calibrated probability: 0.9883


## Result

Training, evaluation, export, and the smoke test now live directly in this notebook.
The old random split, 22-feature block, and XGBoost export are not used.